In [ ]:

!pip install drain3

import pandas as pd
from drain3 import TemplateMiner
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

import os
print(os.listdir('/content/drive/MyDrive'))

log_file_path = "/content/drive/MyDrive/HDFS.log"

template_miner = TemplateMiner()
parsed_results = []

print("Setup complete - ready to parse!")

Mounted at /content/drive
['Programme%20Hanbook%20MDS.pdf', 'Programme%20Hanbook%20MDS.gdoc', 'HDFS.log', 'anomaly_label.csv', 'Colab Notebooks', 'HDFS_structured_sample.csv', 'HDFS_structured_labeled.csv', 'model_comparison_results.csv']
Setup complete - ready to parse!


In [ ]:
test_lines = 5000

with open(log_file_path, "r", errors="ignore") as f:
    for i, line in enumerate(f):
        if i >= test_lines:
            break
        line = line.strip()
        if not line:
            continue
        result = template_miner.add_log_message(line)
        parsed_results.append({
            "raw_log": line,
            "template": result["template_mined"],
            "cluster_id": result["cluster_id"]
        })

df = pd.DataFrame(parsed_results)
print(df.head())

                                             raw_log  \
0  081109 203518 143 INFO dfs.DataNode$DataXceive...   
1  081109 203518 35 INFO dfs.FSNamesystem: BLOCK*...   
2  081109 203519 143 INFO dfs.DataNode$DataXceive...   
3  081109 203519 145 INFO dfs.DataNode$DataXceive...   
4  081109 203519 145 INFO dfs.DataNode$PacketResp...   

                                            template  cluster_id  
0  081109 203518 143 INFO dfs.DataNode$DataXceive...           1  
1  081109 203518 35 INFO dfs.FSNamesystem: BLOCK*...           2  
2  081109 <*> 143 INFO dfs.DataNode$DataXceiver: ...           1  
3  081109 <*> <*> INFO dfs.DataNode$DataXceiver: ...           1  
4  081109 203519 145 INFO dfs.DataNode$PacketResp...           3  


In [ ]:
sample_size = 500000

parsed_results = []

with open(log_file_path, "r", errors="ignore") as f:
    for i, line in enumerate(f):
        if i >= sample_size:
            break
        line = line.strip()
        if not line:
            continue
        result = template_miner.add_log_message(line)
        parsed_results.append({
            "raw_log": line,
            "template": result["template_mined"],
            "cluster_id": result["cluster_id"]
        })

df = pd.DataFrame(parsed_results)
df.to_csv("/content/drive/MyDrive/HDFS_structured_sample.csv", index=False)

print(f"Done! Parsed {len(df)} log lines into {df['cluster_id'].nunique()} unique templates.")

Done! Parsed 500000 log lines into 34 unique templates.


In [ ]:
labels_df = pd.read_csv("/content/drive/MyDrive/anomaly_label.csv")
print(labels_df.head())
print(labels_df['Label'].value_counts())

                    BlockId    Label
0  blk_-1608999687919862906   Normal
1   blk_7503483334202473044   Normal
2  blk_-3544583377289625738  Anomaly
3  blk_-9073992586687739851   Normal
4   blk_7854771516489510256   Normal
Label
Normal     558223
Anomaly     16838
Name: count, dtype: int64


In [ ]:
import re

def extract_block_id(log_line):
    match = re.search(r'(blk_-?\d+)', log_line)
    return match.group(1) if match else None

df['BlockId'] = df['raw_log'].apply(extract_block_id)
print(df[['raw_log', 'BlockId']].head())

                                             raw_log                   BlockId
0  081109 203518 143 INFO dfs.DataNode$DataXceive...  blk_-1608999687919862906
1  081109 203518 35 INFO dfs.FSNamesystem: BLOCK*...  blk_-1608999687919862906
2  081109 203519 143 INFO dfs.DataNode$DataXceive...  blk_-1608999687919862906
3  081109 203519 145 INFO dfs.DataNode$DataXceive...  blk_-1608999687919862906
4  081109 203519 145 INFO dfs.DataNode$PacketResp...  blk_-1608999687919862906


In [ ]:
df_labeled = df.merge(labels_df, on='BlockId', how='left')
print(df_labeled['Label'].value_counts())

Label
Normal     480905
Anomaly     19095
Name: count, dtype: int64


In [ ]:
df_labeled.to_csv("/content/drive/MyDrive/HDFS_structured_labeled.csv", index=False)
print("Saved labeled dataset!")

Saved labeled dataset!


In [ ]:
import pandas as pd
df_labeled = pd.read_csv("/content/drive/MyDrive/HDFS_structured_labeled.csv")
print(df_labeled.shape)
print(df_labeled['Label'].value_counts())

(500000, 5)
Label
Normal     480905
Anomaly     19095
Name: count, dtype: int64


In [ ]:
df_labeled = df_labeled.dropna(subset=['Label'])
print(f"Remaining rows after dropping unmatched: {len(df_labeled)}")

Remaining rows after dropping unmatched: 500000


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000)  # limit to top 1000 words/tokens to keep it manageable
X = vectorizer.fit_transform(df_labeled['template'])

print(f"TF-IDF matrix shape: {X.shape}")  # rows = log lines, columns = features

TF-IDF matrix shape: (500000, 199)


In [ ]:
y = df_labeled['Label'].apply(lambda x: 1 if x == 'Anomaly' else 0)
print(y.value_counts())

Label
0    480905
1     19095
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

Train size: 400000, Test size: 100000


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(),
    "Naive Bayes": MultinomialNB()
}

results = []

for name, model in models.items():
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train

    start_pred = time.time()
    y_pred = model.predict(X_test)
    pred_time = time.time() - start_pred

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "Train Time (s)": train_time,
        "Predict Time (s)": pred_time
    })
    print(f"{name} done.")

results_df = pd.DataFrame(results)
print(results_df)

Logistic Regression done.
Random Forest done.
SVM done.
Naive Bayes done.
                 Model  Accuracy  Precision    Recall  F1-score  \
0  Logistic Regression   0.96405   0.959016  0.061273  0.115186   
1        Random Forest   0.96435   0.953571  0.069914  0.130276   
2                  SVM   0.96436   0.956989  0.069914  0.130307   
3          Naive Bayes   0.96400   0.939759  0.061273  0.115044   

   Train Time (s)  Predict Time (s)  
0        1.018521          0.007205  
1       35.841125          0.330127  
2      609.535198         92.287409  
3        0.039501          0.005506  


In [ ]:
results_df.to_csv("/content/drive/MyDrive/model_comparison_results.csv", index=False)
print("Results saved!")

Results saved!


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time

models_balanced = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight='balanced'),
    "SVM": LinearSVC(class_weight='balanced', max_iter=2000),
    "Naive Bayes": MultinomialNB()
}

results_balanced = []

for name, model in models_balanced.items():
    print(f"Training {name}...")
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train

    start_pred = time.time()
    y_pred = model.predict(X_test)
    pred_time = time.time() - start_pred

    results_balanced.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "Train Time (s)": train_time,
        "Predict Time (s)": pred_time
    })
    print(f"{name} (balanced) done in {train_time:.2f}s.")

results_balanced_df = pd.DataFrame(results_balanced)
print(results_balanced_df)

results_balanced_df.to_csv("/content/drive/MyDrive/model_comparison_results_balanced.csv", index=False)

Training Logistic Regression...
Logistic Regression (balanced) done in 3.55s.
Training Random Forest...
Random Forest (balanced) done in 40.75s.
Training SVM...
SVM (balanced) done in 29.11s.
Training Naive Bayes...
Naive Bayes (balanced) done in 0.04s.
                 Model  Accuracy  Precision    Recall  F1-score  \
0  Logistic Regression   0.67393   0.054034  0.456664  0.096634   
1        Random Forest   0.67401   0.054047  0.456664  0.096655   
2                  SVM   0.67402   0.054049  0.456664  0.096658   
3          Naive Bayes   0.96400   0.939759  0.061273  0.115044   

   Train Time (s)  Predict Time (s)  
0        3.550376          0.008001  
1       40.749558          0.313310  
2       29.106286          0.003671  
3        0.043655          0.005940  


In [ ]:
results_df = pd.read_csv("/content/drive/MyDrive/model_comparison_results.csv")
results_balanced_df = pd.read_csv("/content/drive/MyDrive/model_comparison_results_balanced.csv")

comparison = results_df.merge(results_balanced_df, on="Model", suffixes=("_before", "_after"))
print(comparison[["Model", "Accuracy_before", "Accuracy_after",
                   "Precision_before", "Precision_after",
                   "Recall_before", "Recall_after",
                   "F1-score_before", "F1-score_after"]])

                 Model  Accuracy_before  Accuracy_after  Precision_before  \
0  Logistic Regression          0.96405         0.67393          0.959016   
1        Random Forest          0.96435         0.67401          0.953571   
2                  SVM          0.96436         0.67402          0.956989   
3          Naive Bayes          0.96400         0.96400          0.939759   

   Precision_after  Recall_before  Recall_after  F1-score_before  \
0         0.054034       0.061273      0.456664         0.115186   
1         0.054047       0.069914      0.456664         0.130276   
2         0.054049       0.069914      0.456664         0.130307   
3         0.939759       0.061273      0.061273         0.115044   

   F1-score_after  
0        0.096634  
1        0.096655  
2        0.096658  
3        0.115044  


In [ ]:
# Combine before and after results into one final comparison table
results_df = pd.read_csv("/content/drive/MyDrive/model_comparison_results.csv")
results_balanced_df = pd.read_csv("/content/drive/MyDrive/model_comparison_results_balanced.csv")

final_comparison = results_df.merge(results_balanced_df, on="Model", suffixes=("_before", "_after"))

# Reorder columns for clarity
final_comparison = final_comparison[[
    "Model",
    "Accuracy_before", "Accuracy_after",
    "Precision_before", "Precision_after",
    "Recall_before", "Recall_after",
    "F1-score_before", "F1-score_after",
    "Train Time (s)_before", "Train Time (s)_after",
    "Predict Time (s)_before", "Predict Time (s)_after"
]]

# Save to Drive
final_comparison.to_csv("/content/drive/MyDrive/final_model_comparison.csv", index=False)

print("Saved final comparison!")
print(final_comparison)

Saved final comparison!
                 Model  Accuracy_before  Accuracy_after  Precision_before  \
0  Logistic Regression          0.96405         0.67393          0.959016   
1        Random Forest          0.96435         0.67401          0.953571   
2                  SVM          0.96436         0.67402          0.956989   
3          Naive Bayes          0.96400         0.96400          0.939759   

   Precision_after  Recall_before  Recall_after  F1-score_before  \
0         0.054034       0.061273      0.456664         0.115186   
1         0.054047       0.069914      0.456664         0.130276   
2         0.054049       0.069914      0.456664         0.130307   
3         0.939759       0.061273      0.061273         0.115044   

   F1-score_after  Train Time (s)_before  Train Time (s)_after  \
0        0.096634               1.018521              3.550376   
1        0.096655              35.841125             40.749558   
2        0.096658             609.535198           